<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:46px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Ablation Study · Controlled Comparison · Medical Computer Vision</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Segmentasyon Başarıyı Artırıyor mu?<br>Akciğer-Odaklı ViT için Kontrollü Ablasyon Çalışması</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Aynı bölme, aynı tohum, aynı hiperparametreler — tek değişken: sınıflandırıcıya ulaşan piksellerin anatomik olarak sınırlanıp sınırlanmadığı</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Kol A:</b> Segmentasyonsuz (taban çizgi)</div>
    <div><b>Kol B:</b> Yalnızca RoI kırpma (maskeleme yok)</div>
    <div><b>Kol C:</b> Maske + RoI kırpma (önerilen)</div>
    <div><b>Sınıflandırıcı:</b> ViT-B/16, üç kolda birebir aynı</div>
    <div><b>Değerlendirme:</b> İç test + RSNA + NIH ChestX-ray14</div>
    <div><b>İstatistik:</b> Bootstrap GA · DeLong · McNemar</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Soru:</b> Anatomik segmentasyon sınıflandırma başarımını gerçekten artırıyor mu, yoksa yalnızca modelin <i>nereye baktığını</i> mı değiştiriyor? Bu iki olasılık farklı sonuçlar doğurur ve yalnızca kontrollü bir ablasyonla ayrıştırılabilir. Üçüncü kol (yalnızca kırpma) kritiktir: olası kazancın <i>maskelemeden</i> mi yoksa sadece <i>kadrajlamadan</i> mı geldiğini ayırır — hakemlerin ilk soracağı sorudur.
  </div>
</div>

## Öz

Önceki çalışmada akciğer-odaklı ön-işleme hattı ile eğitilen ViT-B/16, iç test kümesinde
ROC-AUC = 0,998 ve F1 = 0,978 üretmiş; eğitim dağıtımından bağımsız iki yetişkin kümede
sırasıyla AUC = 0,930 (RSNA) ve AUC = 0,702 (NIH) değerlerine ulaşmıştır. Ancak bu
sayılar tek başına **segmentasyonun katkısını ölçmez**, çünkü karşılaştırılacak bir kontrol
kolu yoktur.

Bu notebook, o boşluğu kapatan kontrollü bir ablasyon kurar. Üç kol **aynı veri bölmesi,
aynı rastgelelik tohumu, aynı mimari, aynı artırma ve aynı optimizasyon çizelgesiyle**
sıfırdan eğitilir; kollar arasındaki tek fark, sınıflandırıcıya ulaşan görüntünün nasıl
hazırlandığıdır.

| Kol | Ön-işleme | İzole ettiği etki |
|-----|-----------|-------------------|
| **A · `raw`** | Görüntü doğrudan 224² boyutuna ölçeklenir | Segmentasyon yok — literatür taban çizgisi |
| **B · `roi`** | Segmentasyon **yalnızca** ilgi alanı kutusunu bulmak için kullanılır; piksel silinmez | Kadrajlama (yakınlaştırma) etkisi |
| **C · `lung`** | Akciğer dışı pikseller silinir + aynı ilgi alanına kırpılır | Önerilen yöntemin tamamı |

**B ve C birebir aynı kırpma kutusunu kullanır.** Dolayısıyla B → C farkı yalnızca
maskelemedir; A → B farkı yalnızca kadrajlamadır. Bu ayrıştırma olmadan "segmentasyon
işe yarıyor" iddiası desteklenemez.

### Raporlanan büyüklükler

- **Birincil:** ROC-AUC (eşikten bağımsız), %95 bootstrap güven aralığıyla
- **İkincil:** AP, doğruluk, F1, duyarlılık, özgüllük (eşik 0,50), Brier, ECE
- **Eşleştirilmiş testler:** AUC farkı için **DeLong**, doğruluk farkı için **McNemar**
- **Yorumlanabilirlik:** kol başına akciğer-odak oranı (LFR)

### İçindekiler

| § | Bölüm |
|---|-------|
| 1 | [Deney tasarımı ve kontrol değişkenleri](#s1) |
| 2 | [Veri bölmesinin birebir yeniden üretimi](#s2) |
| 3 | [Segmentasyon modeli](#s3) |
| 4 | [Üç kolun ön-işlemesi](#s4) |
| 5 | [Çevrimdışı önbellek ve önizleme](#s5) |
| 6 | [Eğitim — üç kol, özdeş koşullar](#s6) |
| 7 | [İç test değerlendirmesi](#s7) |
| 8 | [Dış doğrulama: RSNA ve NIH](#s8) |
| 9 | [İstatistiksel karşılaştırma](#s9) |
| 10 | [Sonuç figürleri](#s10) |
| 11 | [Akciğer-odak oranı karşılaştırması](#s11) |
| 12 | [Çıktılar ve yorum rehberi](#s12) |

## 1. Deney tasarımı ve kontrol değişkenleri <a id='s1'></a>

Bir ablasyonun geçerliliği, **sabit tutulanların** eksiksizliğine bağlıdır. Aşağıdaki
değişkenler üç kolda birebir aynıdır ve bu hücrede merkezîleştirilir:

| Sabit tutulan | Değer |
|---|---|
| Veri bölmesi | Aynı dosya listeleri (§2'de imza ile doğrulanır) |
| Rastgelelik tohumu | 42 — her kolun eğitimi öncesinde sıfırlanır |
| Mimari | ViT-B/16, ImageNet ön-eğitimli, son 2 blok + LN + başlık |
| Girdi çözünürlüğü | 224 × 224 × 3 |
| Normalizasyon | ortalama 0,4769 / standart sapma 0,2414 |
| Artırma | HFlip 0,5 · döndürme 8° · öteleme 0,04 · ölçek 0,95–1,05 · jitter 0,15 |
| Kayıp | Çapraz entropi (ceza terimi yok) |
| Optimizasyon | AdamW, lr 1e-4, wd 1e-2, kosinüs → 1e-6, 15 epoch, toplu 16 |
| Model seçimi | En iyi doğrulama F1'i |
| Dış doğrulama örneklemi | Sınıf başına 400, tohum 42 ile seçilmiş aynı görüntüler |

> **Yeniden üretilebilirlik notu.** GPU üzerinde bit düzeyinde aynılık garanti edilemez
> (cuDNN çekirdek seçimi ve kayan nokta toplama sırası). `cudnn.benchmark` kapatılır ve
> tüm tohumlar sabitlenir; kalan değişkenlik AUC ölçeğinde ihmal edilebilir düzeydedir ve
> zaten bootstrap güven aralıkları içinde raporlanır.

### Çalıştırma süresi

Üç kolun eğitimi T4 üzerinde yaklaşık **3–3,5 saat** sürer (kol başına ~65 dk), buna
~15 dk önbellekleme ve ~12 dk dış doğrulama eklenir. Toplam ≈ 4 saat; tek bir Kaggle
oturumuna sığar. Hattı önce doğrulamak isterseniz `QUICK_TEST = True` yapın: küçük bir
alt küme ve 2 epoch ile tüm akış birkaç dakikada uçtan uca çalışır.

In [ ]:
import os, sys, gc, json, time, math, random, hashlib, warnings, types, subprocess
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import Counter, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, f1_score, confusion_matrix,
                             classification_report, brier_score_loss)
from scipy import stats as sp_stats

# ----------------------------- CALISTIRMA AYARLARI -----------------------------
SEED              = 42
QUICK_TEST        = False   # True -> kucuk alt kume + 2 epoch (hat dogrulama kosusu)
EPOCHS            = 15
BATCH_SIZE        = 16
MAX_PER_CLASS_EXT = 400     # dis setlerden sinif basina goruntu
SAVE_CHECKPOINTS  = True
ARMS              = ["raw", "roi", "lung"]
# -------------------------------------------------------------------------------

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def seed_worker(worker_id):
    ws = SEED + worker_id
    np.random.seed(ws); random.seed(ws)

set_seed(SEED)
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "img_size"         : 224,
    "batch_size"       : BATCH_SIZE,
    "epochs"           : EPOCHS,
    "lr"               : 1e-4,
    "weight_decay"     : 1e-2,
    "dropout"          : 0.1,
    "num_classes"      : 2,
    "mean"             : [0.4769, 0.4769, 0.4769],
    "std"              : [0.2414, 0.2414, 0.2414],
    "orig_short_max"   : 512,
    "mask_dilate_frac" : 0.025,
    "roi_pad_frac"     : 0.05,
    "mask_feather"     : 9,
    "fill_mode"        : "mean",
    "raw_mode"         : "resize",   # "resize" (tam gorus alani) | "centercrop"
}

# --- Kollarin kimligi: sabit sira, asla dongusel atama ---
ARM_LABEL = OrderedDict([
    ("raw",  "A · Segmentasyonsuz"),
    ("roi",  "B · Yalnizca RoI kirpma"),
    ("lung", "C · Maske + RoI (onerilen)"),
])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}

# --- Onbellek: /kaggle/temp cikti anligina dahil edilmez ---
_cache_base = "/kaggle/temp/ablation" if os.path.isdir("/kaggle/temp") else "/kaggle/working/_ablation_cache"
CACHE_ROOT  = _cache_base + ("_quick" if QUICK_TEST else "")
WORK       = "/kaggle/working"
os.makedirs(CACHE_ROOT, exist_ok=True)

# --- Grafik stili: cekingen izgara, ince isaretler ---
plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})

print("=" * 62)
print(f"  Cihaz : {device}")
if device.type == "cuda":
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  Torch : {torch.__version__}")
print(f"  Kollar: {', '.join(ARMS)}   |  QUICK_TEST = {QUICK_TEST}")
print(f"  Onbellek: {CACHE_ROOT}")
print("=" * 62)

In [ ]:
# ── Girdilerin otomatik kesfi ─────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_classification_root(root=INPUT):
    '''train/{NORMAL,PNEUMONIA} yapisini tasiyan dizini bulur.'''
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    if not hits:
        return None
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0]

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

BASE_PATH = find_classification_root()
assert BASE_PATH is not None, ("Siniflandirma veri kumesi bulunamadi. "
                               "'chest-xray-pneumonia-balanced-dataset' girdisini ekleyin.")
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH   = os.path.join(BASE_PATH, "val")
TEST_PATH  = os.path.join(BASE_PATH, "test")

RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
NIH_BASE  = find_dir_with("Data_Entry_2017.csv")

print("Bulunan girdiler")
print(f"  Siniflandirma : {BASE_PATH}")
print(f"  RSNA          : {RSNA_BASE}")
print(f"  NIH           : {NIH_BASE}")
if RSNA_BASE is None and NIH_BASE is None:
    print("  UYARI: hicbir dis set bulunamadi - yalnizca ic test raporlanacak.")

## 2. Veri bölmesinin birebir yeniden üretimi <a id='s2'></a>

Ablasyonun ilk koşulu, üç kolun **tam olarak aynı görüntüleri** görmesidir. Bölme,
önceki çalışmadaki kod yolu **aynı sırayla** çalıştırılarak yeniden üretilir:
`random.seed(42)` → doğrulama ve test havuzunun birleştirilmesi → dosya imzasıyla
sızıntı denetimi → sınıf bazında karıştırma → eşit bölme.

Aradaki hiçbir adım `random` durumunu tüketmediğinden üretilen bölme öncekiyle
özdeştir. Bunu doğrulamak için her bölmenin **MD5 imzası** yazdırılır; imzalar
sonraki koşumlarda ve önceki çalışmada aynı çıkmalıdır.

In [ ]:
def file_sig(p):
    return f"{os.path.basename(p)}_{os.path.getsize(p) if os.path.exists(p) else 0}"

def split_fingerprint(samples):
    '''Bolmenin icerik imzasi - koşumlar arasi karsilastirma icin.'''
    key = "|".join(sorted(os.path.basename(p) for p, _ in samples))
    return hashlib.md5(key.encode("utf-8")).hexdigest()[:16]

# --- Onceki calismadaki sira birebir korunur ---
set_seed(SEED)

train_if = datasets.ImageFolder(TRAIN_PATH)
val_if   = datasets.ImageFolder(VAL_PATH)
test_if  = datasets.ImageFolder(TEST_PATH)

CLASS_NAMES  = train_if.classes                 # ['NORMAL', 'PNEUMONIA']
CLASS_TO_IDX = train_if.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
PNEU_IDX     = CLASS_TO_IDX["PNEUMONIA"]

train_samples = list(train_if.samples)
pool = list(val_if.samples) + list(test_if.samples)

train_sigs = set(file_sig(p) for p, _ in train_samples)
before = len(pool)
pool = [(p, l) for p, l in pool if file_sig(p) not in train_sigs]
leak = before - len(pool)

idx_n, idx_p = CLASS_TO_IDX["NORMAL"], CLASS_TO_IDX["PNEUMONIA"]
normal_pool = [(p, l) for p, l in pool if l == idx_n]
pneum_pool  = [(p, l) for p, l in pool if l == idx_p]
random.shuffle(normal_pool); random.shuffle(pneum_pool)
per = min(len(normal_pool), len(pneum_pool)) // 2

val_samples  = normal_pool[:per]      + pneum_pool[:per]
test_samples = normal_pool[per:2*per] + pneum_pool[per:2*per]
random.shuffle(val_samples); random.shuffle(test_samples)

splits_raw = {"train": train_samples, "val": val_samples, "test": test_samples}

if QUICK_TEST:
    rq = random.Random(SEED)
    splits_raw = {k: rq.sample(v, min(len(v), 240)) for k, v in splits_raw.items()}
    print(">>> QUICK_TEST etkin - bolmeler alt orneklendi (imzalar tam kosumla ESLESMEZ).\n")

vp = set(p for p, _ in splits_raw["val"]); tp = set(p for p, _ in splits_raw["test"])
trp = set(p for p, _ in splits_raw["train"])

print("=" * 70)
print("  VERI BOLMESI")
print("=" * 70)
print(f"  Havuz: val({len(val_if.samples)}) + test({len(test_if.samples)}) "
      f"| train sizintisi: {leak}")
for name in ["train", "val", "test"]:
    s = splits_raw[name]; c = Counter(l for _, l in s)
    print(f"  {name.upper():<6}: {len(s):>5} | NORMAL:{c[idx_n]:>4}  PNEUMONIA:{c[idx_p]:>4} "
          f"| imza {split_fingerprint(s)}")
print("-" * 70)
print(f"  val <-> test cakisma : {len(vp & tp)}")
print(f"  train <-> val/test   : {len(trp & (vp | tp))}")
print("=" * 70)

## 3. Segmentasyon modeli <a id='s3'></a>

`ianpan/chest-x-ray-basic` (CheXmask üzerinde eğitilmiş U-Net) yüklenir. Model çıktısının
`argmax`'ı her piksele bir etiket verir: 0 arka plan, **1 sağ akciğer**, **2 sol akciğer**,
3 kalp. B ve C kolları yalnızca 1 ve 2'yi kullanır; kalp ve mediasten kasıtlı olarak dışlanır.

A kolu bu modelden hiç yararlanmaz — ancak akciğer-odak oranını (§11) A kolunda da
ölçebilmek için maske yine de hesaplanıp A'nın koordinat uzayına taşınır. Maske A kolunun
**girdisine** değil, yalnızca **değerlendirmesine** girer.

> İnternet açık olmalıdır (Settings → Internet → On).

In [ ]:
import transformers
from transformers import AutoModel

print("ianpan/chest-x-ray-basic yukleniyor... (transformers", transformers.__version__, ")")

def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()

_orig_finalize = getattr(transformers.modeling_utils.PreTrainedModel,
                         "_finalize_model_loading", None)
try:
    if _orig_finalize is not None:
        def _safe_finalize(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _orig_finalize(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _safe_finalize
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _orig_finalize is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _orig_finalize

print("Segmentasyon modeli hazir (akciger Dice: R=0.957, L=0.948 - model karti degeri).")

## 4. Üç kolun ön-işlemesi <a id='s4'></a>

Tek bir segmentasyon ileri-geçişinden üç kolun girdisi **birlikte** üretilir. Bu hem
hesaplama açısından verimlidir hem de kollar arasında maske tutarlılığını garanti eder.

$$
\text{A (raw)}:\; I \longrightarrow \mathrm{Resize}(224)
$$
$$
\text{B (roi)}:\; I \longrightarrow \mathrm{Crop}(\mathcal{B}) \longrightarrow \mathrm{Resize}(224)
$$
$$
\text{C (lung)}:\; I' = I \odot s + \mu(1-s) \longrightarrow \mathrm{Crop}(\mathcal{B}) \longrightarrow \mathrm{Resize}(224)
$$

Burada $s$ genişletilmiş ve kenarı yumuşatılmış akciğer maskesi, $\mu$ veri kümesi
ortalama grisi, $\mathcal{B}$ ise maskenin sınır kutusundan türetilen **kare ilgi alanı**
kutusudur. **B ve C aynı $\mathcal{B}$ kutusunu paylaşır** — aralarındaki tek fark
$I$ yerine $I'$ kullanılmasıdır. Bu, kadrajlama ile maskeleme etkilerini birbirinden
ayırmanın tek yoludur.

Her kol için ayrıca 224 uzayındaki ikili akciğer maskesi önbelleğe alınır; §11'deki
odak analizinde kullanılır.

In [ ]:
def load_image_any(path, short_max):
    '''PNG/JPG/DICOM -> (rgb_u8, gray_u8); kisa kenari short_max'a indirir.'''
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    '''Sag + sol akciger maskesi; kalp (sinif 3) dislanir.'''
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)


def _center_square(img, S, interp):
    '''Kisa kenari 256'ya olcekle, merkezden S kare kirp.'''
    H, W = img.shape[:2]
    s = 256.0 / min(H, W)
    rz = cv2.resize(img, (max(S, int(round(W * s))), max(S, int(round(H * s)))), interpolation=interp)
    h2, w2 = rz.shape[:2]
    y0, x0 = (h2 - S) // 2, (w2 - S) // 2
    return rz[y0:y0 + S, x0:x0 + S]


def prep_arms(rgb_u8, lung_u8, cfg):
    '''Tek segmentasyon cikisindan uc kolun girdisini uretir.
    Doner: {arm: (img224_u8, mask224_u8, used_bool)}'''
    H, W = lung_u8.shape
    short = min(H, W)
    S = cfg["img_size"]
    out = {}

    # ---------- A: raw (segmentasyondan bagimsiz) ----------
    if cfg["raw_mode"] == "centercrop":
        a_img = _center_square(rgb_u8, S, cv2.INTER_AREA)
        a_msk = _center_square(lung_u8, S, cv2.INTER_NEAREST)
    else:
        a_img = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        a_msk = cv2.resize(lung_u8, (S, S), interpolation=cv2.INTER_NEAREST)
    out["raw"] = (a_img, a_msk.astype(np.uint8), True)

    # ---------- Segmentasyon basarisiz: B ve C icin geri cekilme ----------
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        ones = np.ones((S, S), np.uint8)
        out["roi"] = (fb, ones, False)
        out["lung"] = (fb, ones, False)
        return out

    # ---------- Tolerans katmani ----------
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)

    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]

    fill = (np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
            if cfg["fill_mode"] == "mean" else np.zeros(3, dtype=np.float32))
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)

    # ---------- Ortak RoI kutusu (B ve C ayni kutuyu kullanir) ----------
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)

    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)

    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (S, S),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    return out

print("Uc kollu on-isleme hatti hazir.")

## 5. Çevrimdışı önbellek ve önizleme <a id='s5'></a>

Segmentasyon ağını her epoch'ta çalıştırmak eğitimi kullanılamaz hale getirir; bu nedenle
tüm bölmeler **bir kez** işlenip diske yazılır. Görüntü başına tek segmentasyon geçişi
yapılır ve üç kolun çıktısı birlikte üretilir.

Önbellek `/kaggle/temp` altına yazılır — bu dizin çıktı anlığına dahil edilmediğinden
oturum sonunda binlerce dosyanın işlenmesi gerekmez.

In [ ]:
def cache_paths(arm, split, cls, stem):
    d_img = os.path.join(CACHE_ROOT, arm, split, cls)
    d_msk = os.path.join(CACHE_ROOT, arm, split + "_mask", cls)
    return os.path.join(d_img, stem), os.path.join(d_msk, stem)


def build_cache(split, samples, cfg, log_every=500):
    for arm in ARMS:
        for cls in CLASS_NAMES:
            os.makedirs(os.path.join(CACHE_ROOT, arm, split, cls), exist_ok=True)
            os.makedirs(os.path.join(CACHE_ROOT, arm, split + "_mask", cls), exist_ok=True)

    n_ok = n_fb = n_err = 0
    t0 = time.time()
    for i, (path, label) in enumerate(samples):
        cls = IDX_TO_CLASS[label]
        stem = f"{i:06d}_{os.path.splitext(os.path.basename(path))[0]}.png"
        done = all(os.path.exists(cache_paths(a, split, cls, stem)[0]) and
                   os.path.exists(cache_paths(a, split, cls, stem)[1]) for a in ARMS)
        if done:
            n_ok += 1
            continue
        try:
            rgb, gray = load_image_any(path, cfg["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, cfg)
            for arm in ARMS:
                img, msk, used = arms[arm]
                p_img, p_msk = cache_paths(arm, split, cls, stem)
                Image.fromarray(img).save(p_img)
                Image.fromarray((msk * 255).astype(np.uint8)).save(p_msk)
            n_ok += 1
            n_fb += (0 if arms["lung"][2] else 1)
        except Exception as e:
            n_err += 1
            if n_err <= 5:
                print(f"    hata ({os.path.basename(path)}): {e}")
        if (i + 1) % log_every == 0:
            print(f"    [{split}] {i+1}/{len(samples)}  ({time.time()-t0:.0f} sn)")
    print(f"  [{split}] tamam: {n_ok} kayit | geri cekilme={n_fb} | hata={n_err} "
          f"| {time.time()-t0:.0f} sn")
    return n_fb


print("Uc kol icin cevrimdisi on-isleme basliyor...\n")
fb_total = 0
for name in ["train", "val", "test"]:
    fb_total += build_cache(name, splits_raw[name], CONFIG)
print(f"\nOnbellek hazir: {CACHE_ROOT}  |  toplam geri cekilme: {fb_total}")

In [ ]:
# ── Onizleme: ayni goruntu, uc kol ────────────────────────────────────────
_prev = []
for cls in CLASS_NAMES:
    idx = CLASS_TO_IDX[cls]
    _prev += [(p, idx) for p, l in splits_raw["test"] if l == idx][:2]

fig, axes = plt.subplots(len(_prev), 4, figsize=(11.5, 2.9 * len(_prev)))
fig.suptitle("Ayni radyograftan uc kolun girdisi", fontsize=12, fontweight="bold", y=1.0)
for r, (path, lbl) in enumerate(_prev):
    rgb, gray = load_image_any(path, CONFIG["orig_short_max"])
    lung = lung_mask_ianpan(gray, gray.shape)
    arms = prep_arms(rgb, lung, CONFIG)
    axes[r, 0].imshow(rgb); axes[r, 0].contour(lung, levels=[0.5], colors="#0D8FA2", linewidths=1.2)
    axes[r, 0].set_title(f"{IDX_TO_CLASS[lbl]} - orijinal + kontur", fontsize=8.5)
    axes[r, 0].axis("off")
    for c, arm in enumerate(ARMS, start=1):
        axes[r, c].imshow(arms[arm][0])
        axes[r, c].set_title(ARM_LABEL[arm], fontsize=8.5, color=ARM_COLOR[arm], fontweight="bold")
        axes[r, c].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_01_arms_preview.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Eğitim — üç kol, özdeş koşullar <a id='s6'></a>

Her kolun eğitimi öncesinde tüm tohumlar sıfırlanır ve veri yükleyicinin karıştırma
üreteci sabit tohumla kurulur. Böylece kollar **aynı ağırlık başlangıcını, aynı yığın
sırasını ve aynı artırma rastgeleliğini** paylaşır; geriye kalan tek fark piksellerdir.

In [ ]:
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
    "eval": transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
}


def make_loaders(arm):
    ds, dl = {}, {}
    for split in ["train", "val", "test"]:
        tf = data_transforms["train" if split == "train" else "eval"]
        ds[split] = datasets.ImageFolder(os.path.join(CACHE_ROOT, arm, split), tf)
        assert ds[split].classes == CLASS_NAMES, f"{arm}/{split}: sinif sirasi uyusmazligi"
        g = torch.Generator(); g.manual_seed(SEED)
        dl[split] = DataLoader(ds[split], batch_size=CONFIG["batch_size"],
                               shuffle=(split == "train"), num_workers=2, pin_memory=True,
                               generator=g if split == "train" else None,
                               worker_init_fn=seed_worker)
    return ds, dl


def build_vit(fine_tune_last_n=2):
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.encoder.layers[-fine_tune_last_n:].parameters():
        p.requires_grad = True
    for p in m.encoder.ln.parameters():
        p.requires_grad = True
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(CONFIG["dropout"]),
                                 nn.Linear(in_f, CONFIG["num_classes"]))
    return m.to(device)


def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, preds, labels, probs = 0.0, [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            probs.extend(torch.softmax(out, 1)[:, 1].detach().cpu().numpy())
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    N = len(loader.dataset)
    acc = float((np.array(preds) == np.array(labels)).mean())
    return (loss_sum / N, f1_score(labels, preds, zero_division=0), acc,
            np.array(labels), np.array(preds), np.array(probs))


def train_arm(arm, epochs=None):
    epochs = epochs or CONFIG["epochs"]
    set_seed(SEED)                       # <-- her kol icin ozdes baslangic
    ds, dl = make_loaders(arm)
    model = build_vit(fine_tune_last_n=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    hist = {k: [] for k in ["tr_loss", "tr_f1", "vl_loss", "vl_f1", "vl_acc"]}
    best_f1, best_wt, t0 = 0.0, None, time.time()
    print(f"\n{'='*70}\n  KOL: {arm}  ({ARM_LABEL[arm]})  |  {epochs} epoch  "
          f"|  n_train = {len(ds['train'])}\n{'='*70}")
    for ep in range(epochs):
        tr_loss, tr_f1, _, _, _, _ = run_epoch(model, dl["train"], criterion, optimizer)
        vl_loss, vl_f1, vl_acc, _, _, _ = run_epoch(model, dl["val"], criterion, None)
        scheduler.step()
        for k, v in zip(["tr_loss", "tr_f1", "vl_loss", "vl_f1", "vl_acc"],
                        [tr_loss, tr_f1, vl_loss, vl_f1, vl_acc]):
            hist[k].append(v)
        star = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            best_wt = {k: v.detach().clone() for k, v in model.state_dict().items()}
            star = " *"
        print(f"  Ep{ep+1:3d} | TR kayip {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"VAL kayip {vl_loss:.4f} F1 {vl_f1:.4f} acc {vl_acc:.4f}{star}")
    if best_wt:
        model.load_state_dict(best_wt)
    print(f"  -> en iyi Val F1 = {best_f1:.4f}  ({time.time()-t0:.0f} sn)")
    return model, hist, best_f1, ds, dl

In [ ]:
EPOCHS_RUN = 2 if QUICK_TEST else CONFIG["epochs"]

MODELS, HIST, BESTF1, DSETS, DLOAD = {}, {}, {}, {}, {}
t_all = time.time()
for arm in ARMS:
    m, h, b, ds, dl = train_arm(arm, epochs=EPOCHS_RUN)
    MODELS[arm], HIST[arm], BESTF1[arm], DSETS[arm], DLOAD[arm] = m, h, b, ds, dl
    gc.collect(); torch.cuda.empty_cache()

print(f"\n{'='*70}")
print(f"  Tum kollarin egitimi tamamlandi ({(time.time()-t_all)/60:.1f} dk)")
for arm in ARMS:
    print(f"    {arm:<5} en iyi Val F1 = {BESTF1[arm]:.4f}")
print("=" * 70)

In [ ]:
# ── Egitim egrileri: uc kol ustuste ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
ep = range(1, len(HIST[ARMS[0]]["tr_loss"]) + 1)
for arm in ARMS:
    axes[0].plot(ep, HIST[arm]["vl_loss"], "-o", ms=3.5, lw=2,
                 color=ARM_COLOR[arm], label=ARM_LABEL[arm])
    axes[1].plot(ep, HIST[arm]["vl_f1"], "-o", ms=3.5, lw=2,
                 color=ARM_COLOR[arm], label=ARM_LABEL[arm])
axes[0].set_title("Dogrulama kaybi"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Dogrulama F1"); axes[1].set_xlabel("Epoch")
for a in axes:
    a.yaxis.grid(True, alpha=.55); a.set_axisbelow(True)
axes[1].legend(loc="lower right", fontsize=8.5)
fig.suptitle("Ozdes kosullar altinda uc kolun egitim seyri", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_02_training.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. İç test değerlendirmesi <a id='s7'></a>

Her kol, kendi ön-işlemesinden geçmiş **aynı test görüntüleri** üzerinde değerlendirilir.
Tahmin olasılıkları saklanır; §9'daki eşleştirilmiş testler bunları kullanır.

In [ ]:
criterion = nn.CrossEntropyLoss()
PRED = {}          # PRED[dataset][arm] = dict(y_true, y_prob)

PRED["internal"] = {}
rows = []
for arm in ARMS:
    _, f1v, accv, yt, yp, pr = run_epoch(MODELS[arm], DLOAD[arm]["test"], criterion, None)
    PRED["internal"][arm] = {"y_true": yt, "y_prob": pr}
    fpr, tpr, _ = roc_curve(yt, pr)
    rows.append({"kol": arm, "N": len(yt), "AUC": auc(fpr, tpr),
                 "AP": average_precision_score(yt, pr), "Acc": accv, "F1": f1v})
    print(f"[{arm:<5}] Acc {accv:.4f} | F1 {f1v:.4f} | AUC {rows[-1]['AUC']:.4f}")

df_int = pd.DataFrame(rows).set_index("kol")
print("\n" + "=" * 62)
print("  IC TEST OZETI")
print("=" * 62)
print(df_int.round(4).to_string())
print("=" * 62)

## 8. Dış doğrulama: RSNA ve NIH <a id='s8'></a>

Asıl sınav burasıdır. Segmentasyonun değeri, aynı dağıtımdaki doğrulukta değil,
**dağıtım kayması altındaki dayanıklılıkta** aranmalıdır — kısayolları ortadan kaldıran
bir ön-işlemenin beklenen faydası tam olarak budur.

Örneklem, önceki dış doğrulama çalışmasıyla **aynı tohumla** kurulur; dolayısıyla
görüntüler birebir aynıdır ve C kolunun sonuçları daha önce raporlanan değerlerle
doğrudan karşılaştırılabilir. Her görüntü için tek segmentasyon geçişi yapılır, üç kolun
girdisi üretilir ve üç model sırayla değerlendirilir.

In [ ]:
def build_rsna_items(base, k, seed=SEED):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(os.path.join(img_dir, p + ".dcm"), PNEU_IDX) for p in pos[:k]]
    items += [(os.path.join(img_dir, p + ".dcm"), 1 - PNEU_IDX) for p in neg[:k]]
    return items


def build_nih_items(base, k, seed=SEED):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    labels = df["Finding Labels"].astype(str)
    is_pneu   = labels.apply(lambda s: "Pneumonia" in s.split("|"))
    is_normal = labels.apply(lambda s: s.strip() == "No Finding")
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    def paths_for(mask_):
        return [index[n] for n in df[mask_]["Image Index"].tolist() if n in index]
    pos, neg = paths_for(is_pneu), paths_for(is_normal)
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    items  = [(p, PNEU_IDX) for p in pos[:k]]
    items += [(p, 1 - PNEU_IDX) for p in neg[:k]]
    return items


eval_tf = data_transforms["eval"]
K_EXT = 60 if QUICK_TEST else MAX_PER_CLASS_EXT

EXTERNAL = []
if RSNA_BASE:
    EXTERNAL.append(("RSNA", build_rsna_items(RSNA_BASE, K_EXT)))
if NIH_BASE:
    EXTERNAL.append(("NIH", build_nih_items(NIH_BASE, K_EXT)))
for nm, it in EXTERNAL:
    print(f"{nm:<5}: {len(it)} goruntu "
          f"({sum(l == PNEU_IDX for _, l in it)} pnomoni / {sum(l != PNEU_IDX for _, l in it)} normal)")

In [ ]:
@torch.inference_mode()
def external_inference(name, items):
    '''Her goruntu icin tek segmentasyon -> uc kolun girdisi -> uc model.'''
    for arm in ARMS:
        MODELS[arm].eval()
    y_true, probs, used_flags = [], {a: [] for a in ARMS}, []
    t0 = time.time()
    for i, (path, label) in enumerate(items):
        try:
            rgb, gray = load_image_any(path, CONFIG["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, CONFIG)
        except Exception:
            continue
        y_true.append(label)
        used_flags.append(arms["lung"][2])
        for arm in ARMS:
            x = eval_tf(Image.fromarray(arms[arm][0])).unsqueeze(0).to(device)
            probs[arm].append(torch.softmax(MODELS[arm](x), 1)[0, PNEU_IDX].item())
        if (i + 1) % 200 == 0:
            print(f"    [{name}] {i+1}/{len(items)}  ({time.time()-t0:.0f} sn)")
    fb = 100.0 * (1.0 - np.mean(used_flags)) if used_flags else 0.0
    print(f"  [{name}] bitti: {len(y_true)} goruntu | {time.time()-t0:.0f} sn "
          f"| geri cekilme %{fb:.1f}")
    yt = np.array(y_true)
    return {arm: {"y_true": yt, "y_prob": np.array(probs[arm])} for arm in ARMS}


for name, items in EXTERNAL:
    print(f"\n>>> {name} cikarimi ({len(items)} goruntu, uc kol)...")
    PRED[name] = external_inference(name, items)

In [ ]:
# ── Tum kume x kol metrik tablosu ────────────────────────────────────────
def expected_calibration_error(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); ece, N = 0.0, len(y_true)
    for i in range(n_bins):
        m = (y_prob > edges[i]) & (y_prob <= edges[i + 1])
        if m.sum() == 0:
            continue
        ece += (m.sum() / N) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece


def metrics_block(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn + 1e-9); spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9); f1 = 2 * prec * sens / (prec + sens + 1e-9)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {"N": len(y_true), "AUC": auc(fpr, tpr),
            "AP": average_precision_score(y_true, y_prob),
            "Acc": float((y_pred == y_true).mean()), "F1": f1,
            "Duyarlilik": sens, "Ozgulluk": spec,
            "Brier": brier_score_loss(y_true, y_prob),
            "ECE": expected_calibration_error(y_true, y_prob)}


DATASETS = ["internal"] + [n for n, _ in EXTERNAL]
DS_LABEL = {"internal": "Ic test", "RSNA": "RSNA (yetiskin)", "NIH": "NIH ChestX-ray14"}

rows = []
for dsname in DATASETS:
    for arm in ARMS:
        d = PRED[dsname][arm]
        rows.append({"kume": DS_LABEL[dsname], "kol": arm, **metrics_block(d["y_true"], d["y_prob"])})
df_all = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("=" * 104)
print("  TUM SONUCLAR  (esik = 0,50)")
print("=" * 104)
print(df_all.to_string(index=False))
print("=" * 104)

## 9. İstatistiksel karşılaştırma <a id='s9'></a>

İki AUC arasındaki farkın anlamlı olup olmadığı gözle karara bağlanamaz. Üç araç kullanılır:

**Bootstrap güven aralığı.** Test kümesi sınıf-katmanlı olarak 2000 kez yeniden
örneklenir; her kolun AUC'si ve kollar arası **eşleştirilmiş fark** için yüzdelik %95
aralığı hesaplanır. Aynı yeniden örnekleme indeksleri tüm kollara uygulanır — fark
dağılımı bu sayede eşleştirilmiş olur.

**DeLong testi.** Aynı örneklem üzerinde hesaplanan iki ROC eğrisinin ilişkili
olduğunu dikkate alan standart test. AUC farkı için asimptotik $z$ ve $p$ üretir.

**McNemar testi.** Eşik 0,50'de ikili doğru/yanlış kararların eşleştirilmiş
karşılaştırması; yalnızca iki kolun **farklılaştığı** örnekleri kullanır.

In [ ]:
# ── DeLong (Sun & Xu, 2014 hizli uygulama) ───────────────────────────────
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T + 1
    return T2


def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _midrank(pos[r, :])
        ty[r, :] = _midrank(neg[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    if k == 1:
        sx = np.array([[float(sx)]]); sy = np.array([[float(sy)]])
    return aucs, sx / m + sy / n


def delong_test(y_true, p1, p2):
    '''Iliskili iki ROC egrisi icin AUC farki testi. Doner: (auc1, auc2, z, p).'''
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")     # pozitifler once
    m = int(y.sum())
    preds = np.vstack((np.asarray(p1), np.asarray(p2)))[:, order]
    aucs, cov = _fast_delong(preds, m)
    l = np.array([[1.0, -1.0]])
    var = float(l.dot(cov).dot(l.T))
    if var <= 0:
        # varyans sifir: tahminciler ozdes -> fark yok
        return aucs[0], aucs[1], 0.0, 1.0
    z = float((aucs[0] - aucs[1]) / np.sqrt(var))
    return aucs[0], aucs[1], z, float(2 * (1 - sp_stats.norm.cdf(abs(z))))


# ── Eslestirilmis bootstrap ──────────────────────────────────────────────
def paired_bootstrap(y_true, prob_dict, n_boot=2000, seed=SEED):
    '''Sinif-katmanli yeniden orneklemeyle AUC ve kollar arasi fark dagilimi.'''
    rng = np.random.default_rng(seed)
    y = np.asarray(y_true)
    idx_pos = np.where(y == 1)[0]; idx_neg = np.where(y == 0)[0]
    arms = list(prob_dict.keys())
    out = {a: np.empty(n_boot) for a in arms}
    for b in range(n_boot):
        ii = np.concatenate([rng.choice(idx_pos, len(idx_pos), replace=True),
                             rng.choice(idx_neg, len(idx_neg), replace=True)])
        yb = y[ii]
        for a in arms:
            fpr, tpr, _ = roc_curve(yb, np.asarray(prob_dict[a])[ii])
            out[a][b] = auc(fpr, tpr)
    return out


def mcnemar(y_true, p1, p2, thr=0.5):
    c1 = ((np.asarray(p1) >= thr).astype(int) == y_true)
    c2 = ((np.asarray(p2) >= thr).astype(int) == y_true)
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return b, c, 1.0
    p = float(sp_stats.binomtest(b, b + c, 0.5).pvalue)
    return b, c, p

print("Istatistik araclari hazir: DeLong, eslestirilmis bootstrap, McNemar.")

In [ ]:
N_BOOT = 300 if QUICK_TEST else 2000
CI, BOOT = {}, {}

print(f"Bootstrap ({N_BOOT} tekrar) hesaplaniyor...\n")
for dsname in DATASETS:
    d = PRED[dsname]
    BOOT[dsname] = paired_bootstrap(d[ARMS[0]]["y_true"],
                                    {a: d[a]["y_prob"] for a in ARMS}, n_boot=N_BOOT)
    CI[dsname] = {a: (float(np.percentile(BOOT[dsname][a], 2.5)),
                      float(np.percentile(BOOT[dsname][a], 97.5))) for a in ARMS}
    print(f"  {DS_LABEL[dsname]}: tamam")

# --- Eslestirilmis karsilastirmalar ---
PAIRS = [("lung", "raw"), ("lung", "roi"), ("roi", "raw")]
rows = []
for dsname in DATASETS:
    d = PRED[dsname]
    yt = d[ARMS[0]]["y_true"]
    for a, b in PAIRS:
        auc_a, auc_b, z, p = delong_test(yt, d[a]["y_prob"], d[b]["y_prob"])
        diff = BOOT[dsname][a] - BOOT[dsname][b]
        nb, nc, p_mc = mcnemar(yt, d[a]["y_prob"], d[b]["y_prob"])
        rows.append({
            "kume": DS_LABEL[dsname], "karsilastirma": f"{a} - {b}",
            "dAUC": auc_a - auc_b,
            "GA alt": float(np.percentile(diff, 2.5)),
            "GA ust": float(np.percentile(diff, 97.5)),
            "DeLong z": z, "DeLong p": p,
            "McNemar b/c": f"{nb}/{nc}", "McNemar p": p_mc,
        })
df_stat = pd.DataFrame(rows)

print("\n" + "=" * 112)
print("  ESLESTIRILMIS KARSILASTIRMALAR  (dAUC > 0 -> ilk kol ustun)")
print("=" * 112)
print(df_stat.to_string(index=False))
print("=" * 112)
print("  Not: GA aralığı 0'ı içermiyorsa fark %5 düzeyinde anlamlıdır.")

## 10. Sonuç figürleri <a id='s10'></a>

Birincil figür, kümeler arasında kol başına ROC-AUC'yi %95 bootstrap güven aralıklarıyla
gösterir. Her çubuğun değeri doğrudan etiketlenir; renk tek başına kimlik taşımaz.

In [ ]:
# ── Figur 1: kume x kol AUC (bootstrap GA ile) ───────────────────────────
fig, ax = plt.subplots(figsize=(9.8, 4.8))
xs = np.arange(len(DATASETS)); w = 0.26
for k, arm in enumerate(ARMS):
    vals = np.array([df_all[(df_all.kume == DS_LABEL[d]) & (df_all.kol == arm)]["AUC"].iloc[0]
                     for d in DATASETS])
    lo = np.array([CI[d][arm][0] for d in DATASETS])
    hi = np.array([CI[d][arm][1] for d in DATASETS])
    pos = xs + (k - 1) * w
    ax.bar(pos, vals, width=w * 0.88, color=ARM_COLOR[arm], edgecolor="white",
           linewidth=1.6, label=ARM_LABEL[arm], zorder=3)
    ax.errorbar(pos, vals, yerr=[vals - lo, hi - vals], fmt="none",
                ecolor="#2B3438", elinewidth=1.1, capsize=3.5, zorder=4)
    for x, v, h in zip(pos, vals, hi):
        ax.text(x, h + 0.012, f"{v:.3f}", ha="center", va="bottom",
                fontsize=8.6, color="#1B2327", zorder=5)

ax.axhline(0.5, color="#8A9499", ls=":", lw=1, zorder=2)
ax.text(len(DATASETS) - 0.52, 0.505, "rastgele", fontsize=8, color="#8A9499", va="bottom")
ax.set_xticks(xs); ax.set_xticklabels([DS_LABEL[d] for d in DATASETS])
ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.45, 1.06)
ax.yaxis.grid(True, alpha=.55); ax.set_axisbelow(True)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=9)
fig.suptitle("Segmentasyonun katkisi: kol basina ROC-AUC (%95 bootstrap GA)",
             fontsize=12, fontweight="bold", y=1.12)
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_03_auc_by_arm.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur 2: ROC egrileri, kume basina panel ─────────────────────────────
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.4 * len(DATASETS), 4.2))
if len(DATASETS) == 1:
    axes = [axes]
for ax, dsname in zip(axes, DATASETS):
    for arm in ARMS:
        d = PRED[dsname][arm]
        fpr, tpr, _ = roc_curve(d["y_true"], d["y_prob"])
        ax.plot(fpr, tpr, lw=2, color=ARM_COLOR[arm],
                label=f"{ARM_LABEL[arm]} — {auc(fpr, tpr):.3f}")
    ax.plot([0, 1], [0, 1], ls=":", lw=1, color="#8A9499")
    ax.set_title(DS_LABEL[dsname]); ax.set_xlabel("1 - Ozgulluk")
    ax.set_ylabel("Duyarlilik"); ax.grid(alpha=.45); ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=7.8)
fig.suptitle("ROC egrileri — uc kol, uc kume", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_04_roc.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur 3: eslestirilmis AUC farki (etki buyuklugu) ────────────────────
fig, ax = plt.subplots(figsize=(9.0, 3.6))
labels, ys = [], []
for dsname in DATASETS:
    for a, b in PAIRS:
        r = df_stat[(df_stat.kume == DS_LABEL[dsname]) &
                    (df_stat.karsilastirma == f"{a} - {b}")].iloc[0]
        labels.append(f"{DS_LABEL[dsname]}\n{a} - {b}")
        ys.append((r["dAUC"], r["GA alt"], r["GA ust"], r["DeLong p"], a))
y = np.arange(len(labels))
for i, (d, lo, hi, p, a) in enumerate(ys):
    ax.plot([lo, hi], [i, i], lw=2, color=ARM_COLOR[a], solid_capstyle="round", zorder=3)
    ax.plot([d], [i], "o", ms=7, color=ARM_COLOR[a], mec="white", mew=1.4, zorder=4)
    star = "*" if (not np.isnan(p) and p < 0.05) else ""
    ax.text(hi + 0.008, i, f"{d:+.3f}{star}", va="center", fontsize=8.4, color="#1B2327")
ax.axvline(0, color="#5A686F", lw=1.1, zorder=2)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("AUC farki (%95 eslestirilmis bootstrap GA)")
ax.xaxis.grid(True, alpha=.5); ax.set_axisbelow(True)
ax.invert_yaxis()
fig.suptitle("Etki buyuklugu — 0'i icermeyen aralik anlamli fark demektir  (* DeLong p < 0,05)",
             fontsize=10.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_05_effect.png"), dpi=160, bbox_inches="tight")
plt.show()

## 11. Akciğer-odak oranı karşılaştırması <a id='s11'></a>

Başarım tek başına yeterli değildir: segmentasyon başarımı değiştirmese bile modelin
**nereye baktığını** değiştiriyorsa bu kendi başına bir bulgudur. Her kolun dikkat haritası
Attention Rollout ile çıkarılır ve o kolun kendi koordinat uzayındaki akciğer maskesiyle
karşılaştırılır.

$$\mathrm{LFR}=\frac{\sum_{u \in \Omega} S(u)\,L(u)}{\sum_{u \in \Omega} S(u)},\qquad
\Omega=\{u: S(u)>0{,}20\}$$

**Yorum uyarısı.** C kolunda girdi zaten yalnızca akciğerden ibarettir; yüksek LFR kısmen
tasarım gereğidir. Asıl bilgi veren karşılaştırma **A kolunun LFR'sidir**: model tüm
toraksı görebildiğinde dikkatini akciğere veriyor mu, yoksa akciğer dışına mı kayıyor?

In [ ]:
class AttentionRollout:
    '''CLS -> yama dikkat akisi (Abnar & Zuidema, 2020).'''
    def __init__(self, model):
        self.model = model; self.attn_maps = []; self._hooks = []
        for layer in self.model.encoder.layers:
            sa = layer.self_attention
            orig = sa.forward
            def make_patched(orig_fwd):
                def patched(self_mod, *a, **kw):
                    kw["need_weights"] = True
                    kw["average_attn_weights"] = False
                    return orig_fwd(*a, **kw)
                return patched
            sa.forward = types.MethodType(make_patched(orig), sa)
            def make_hook(obj):
                def hook(m, inp, out):
                    if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
                        obj.attn_maps.append(out[1].detach().cpu())
                return hook
            self._hooks.append(sa.register_forward_hook(make_hook(self)))

    def remove(self):
        for h in self._hooks:
            h.remove()

    def compute(self, img_tensor):
        self.attn_maps = []; self.model.eval()
        with torch.no_grad():
            self.model(img_tensor.unsqueeze(0).to(device))
        if not self.attn_maps:
            raise RuntimeError("Dikkat matrisi yakalanamadi.")
        N = self.attn_maps[0].size(-1)
        result = torch.eye(N)
        for attn in self.attn_maps:
            a = attn.squeeze(0)
            if a.dim() == 3:
                a = a.mean(0)
            a = a + torch.eye(N)
            a = a / (a.sum(-1, keepdim=True) + 1e-8)
            result = torch.matmul(a, result)
        m = result[0, 1:].numpy()
        n = int(round(m.size ** 0.5))
        m = m.reshape(n, n)
        m = (m - m.min()) / (m.max() - m.min() + 1e-8)
        m = cv2.resize(m.astype(np.float32), (CONFIG["img_size"], CONFIG["img_size"]))
        return (m - m.min()) / (m.max() - m.min() + 1e-8)


def lung_focus_ratio(sal, mask, tau=0.2):
    s = np.asarray(sal, np.float32).ravel(); m = np.asarray(mask, np.float32).ravel()
    hi = s > tau
    if not hi.any() or s[hi].sum() < 1e-8:
        return float("nan")
    return float((s[hi] * m[hi]).sum() / s[hi].sum())


MAX_LFR = 40 if QUICK_TEST else 120
LFR = {}
for arm in ARMS:
    ro = AttentionRollout(MODELS[arm])
    ds = DSETS[arm]["test"]
    order = list(range(len(ds.samples))); random.Random(SEED).shuffle(order)
    vals = []
    for i in order[:MAX_LFR]:
        path, lbl = ds.samples[i]
        try:
            img_t, _ = ds[i]
            sal = ro.compute(img_t)
            mp = os.path.join(CACHE_ROOT, arm, "test_mask",
                              IDX_TO_CLASS[lbl], os.path.basename(path))
            mask = (np.asarray(Image.open(mp).convert("L")) > 127).astype(np.float32)
            v = lung_focus_ratio(sal, mask)
            if not np.isnan(v):
                vals.append(v)
        except Exception:
            continue
    ro.remove()
    LFR[arm] = np.array(vals)
    print(f"[{arm:<5}] n={len(vals):>3}  LFR = {np.mean(vals):.3f} +/- {np.std(vals):.3f}")

fig, ax = plt.subplots(figsize=(7.2, 3.8))
means = [LFR[a].mean() for a in ARMS]
sds   = [LFR[a].std() for a in ARMS]
xs = np.arange(len(ARMS))
ax.bar(xs, means, width=0.5, color=[ARM_COLOR[a] for a in ARMS],
       edgecolor="white", linewidth=1.6, zorder=3)
ax.errorbar(xs, means, yerr=sds, fmt="none", ecolor="#2B3438",
            elinewidth=1.1, capsize=4, zorder=4)
for x, m, s in zip(xs, means, sds):
    ax.text(x, m + s + 0.02, f"{m:.3f}", ha="center", fontsize=9, color="#1B2327")
ax.set_xticks(xs); ax.set_xticklabels([ARM_LABEL[a] for a in ARMS], fontsize=8.5)
ax.set_ylabel("Akciger-odak orani (LFR)"); ax.set_ylim(0, 1.12)
ax.yaxis.grid(True, alpha=.55); ax.set_axisbelow(True)
fig.suptitle("Model nereye bakiyor? Kol basina akciger-odak orani",
             fontsize=11.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "abl_fig_06_lfr.png"), dpi=160, bbox_inches="tight")
plt.show()

## 12. Çıktılar ve yorum rehberi <a id='s12'></a>

Tüm sayısal sonuçlar CSV ve NPZ olarak dışa aktarılır; figürler bu dosyalardan
notebook'u yeniden çalıştırmadan üretilebilir.

In [ ]:
df_all.to_csv(os.path.join(WORK, "ablation_metrics.csv"), index=False)
df_stat.to_csv(os.path.join(WORK, "ablation_pairwise_tests.csv"), index=False)

ci_rows = [{"kume": DS_LABEL[d], "kol": a,
            "AUC_GA_alt": CI[d][a][0], "AUC_GA_ust": CI[d][a][1]}
           for d in DATASETS for a in ARMS]
pd.DataFrame(ci_rows).to_csv(os.path.join(WORK, "ablation_auc_ci.csv"), index=False)

np.savez_compressed(
    os.path.join(WORK, "ablation_predictions.npz"),
    **{f"{d}__{a}__{k}": PRED[d][a][k]
       for d in DATASETS for a in ARMS for k in ("y_true", "y_prob")},
    **{f"lfr__{a}": LFR[a] for a in ARMS},
)

summary = {
    "seed": SEED, "epochs": EPOCHS_RUN, "quick_test": QUICK_TEST,
    "config": CONFIG, "arms": ARMS,
    "split_fingerprint": {k: split_fingerprint(v) for k, v in splits_raw.items()},
    "best_val_f1": {a: float(BESTF1[a]) for a in ARMS},
    "lfr_mean": {a: float(LFR[a].mean()) for a in ARMS},
}
with open(os.path.join(WORK, "ablation_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

if SAVE_CHECKPOINTS:
    for arm in ARMS:
        torch.save({"model_state_dict": MODELS[arm].state_dict(), "arch": "vit_b_16",
                    "arm": arm, "config": CONFIG, "class_to_idx": CLASS_TO_IDX,
                    "best_val_f1": float(BESTF1[arm])},
                   os.path.join(WORK, f"ablation_vit_{arm}.pth"))

print("Kaydedilenler:")
for f in sorted(os.listdir(WORK)):
    if f.startswith(("ablation", "abl_fig")):
        print(f"  {f:<38} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.1f} MB")

### Sonuçların nasıl okunacağı

Bu ablasyonun cevaplayacağı soru tek bir sayı değil, bir **örüntüdür**. Olası dört sonuç
ve her birinin ne anlama geldiği:

**1. İç testte üç kol benzer, dış doğrulamada C açık ara önde.**
Beklenen ve tezi en güçlü destekleyen sonuçtur. Aynı dağıtımda kısayollar da doğru cevabı
üretir; ayrışma ancak kısayollar geçersizleştiğinde görünür. Bu örüntü, segmentasyonun
katkısının *doğruluk* değil *dayanıklılık* olduğunu gösterir — makalede vurgulanması
gereken çerçeve budur.

**2. C ≈ B > A.** Kazanç maskelemeden değil, kadrajlamadan (akciğerin çerçeveyi
doldurması, etkin çözünürlüğün artması) geliyordur. Bu durumda "anatomik maskeleme"
iddiası zayıflar; katkı bir ilgi alanı normalizasyonu olarak yeniden çerçevelenmelidir.
B kolunun varlık nedeni tam olarak bu ayrımı yapabilmektir.

**3. A ≥ C.** Segmentasyon başarımı düşürüyordur. En olası nedenler: maskenin periferik
patolojiyi kırpması, segmentasyonun yetişkin görüntülerdeki hatası, veya akciğer dışı
bağlamın (kardiyotorasik oran, mediastinal genişlik) gerçek tanısal bilgi taşıması.
Bu da yayınlanabilir bir bulgudur — negatif sonuç, dürüstçe raporlandığında değerlidir.

**4. Farklar güven aralıklarının içinde kalıyor.** Örneklem büyüklüğü mevcut etkiyi
saptamaya yetmiyordur. `MAX_PER_CLASS_EXT` artırılmalı veya birden çok tohumla tekrar
edilip ortalama alınmalıdır.

> **Raporlamada dikkat.** DeLong ve McNemar tek bir karşılaştırma için tasarlanmıştır;
> burada küme başına üç karşılaştırma yapılmaktadır. Makalede birincil hipotezi (C - A,
> dış doğrulamada) önceden belirtip diğerlerini keşifsel olarak sunmak veya çoklu
> karşılaştırma düzeltmesi (Holm) uygulamak gerekir.

> **Kısıt.** Her kol tek bir tohumla bir kez eğitilmiştir. Eğitim rastgeleliğinin etkisini
> ayrıştırmak için ideal olan, kol başına 3–5 tohumla tekrar edip ortalama ± standart
> sapma raporlamaktır; bu, çalıştırma süresini tohum sayısı kadar artırır.